In [53]:
from dotenv import load_dotenv

load_dotenv()

True

In [54]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """
    Use this when user asks you about what movies are popular these days.
    In this URL, popular movies are listed.
    """
    url = f"{BASE_URL}/movies"
    response = requests.get(url)
    return response.json()

def get_movie_details(id):
    """
    If a user asks which movie corresponds to a given ID,
    use this function to retrieve the movie information.
    """
    url = f"{BASE_URL}/movies/{id}"
    response = requests.get(url)
    return response.json()

def get_movie_credits(id):
    """
    If a user asks who appears in the movie associated with a given ID,
    use this function to retrieve the cast and crew information.
    """
    url = f"{BASE_URL}/movies/{id}/credits"
    response = requests.get(url)
    return response.json()


In [55]:
tools = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Use this when the user asks what movies are popular these days.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Use this when the user asks which movie corresponds to a given movie ID.",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Use this when the user asks who appears in the movie for a given ID. Return cast and crew.",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
]

In [68]:
from openai import OpenAI
import json

client = OpenAI()

def ask_to_ai(question):
    response = client.responses.create(
        model = "gpt-4o-mini",
        input = question,
        tools = tools,
    )

    def check_function(response):
        print("호출된 함수 이름 : ", response.output[0].name, "\n")
        print("호출된 함수의 인자 : ", response.output[0].arguments or "{}", "\n")

    check_function(response)

    tool_outputs = []

    for item in response.output:
        if item.type != "function_call":
            continue

        args = json.loads(item.arguments or "{}")
        name = item.name

        if name == "get_popular_movies":
            result = get_popular_movies()

        elif name == "get_movie_details":
            movie_id = int(args["id"])
            result = get_movie_details(movie_id)

        elif name == "get_movie_credits":
            movie_id = int(args["id"])
            result = get_movie_credits(movie_id)

        tool_outputs.append({
            "type":"function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result, ensure_ascii=False)
        })

    if not tool_outputs:
        return response.output_text
    
    response2 = client.responses.create(
        model = "gpt-4o-mini",
        previous_response_id=response.id,
        input = tool_outputs,
    )

    return response2.output_text

In [70]:
questions = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for q in questions:
    print("질문 : ", q, "\n")
    print("답변 : ", ask_to_ai(q), "\n")
    print("---------"* 10, "\n")

질문 :  지금 인기 있는 영화가 무엇인지 알려줘 

호출된 함수 이름 :  get_popular_movies 

호출된 함수의 인자 :  {} 

답변 :  현재 인기 있는 영화 몇 가지를 소개해 드릴게요:

1. **Mercy**
   - **개요**: 미래의 한 형사가 아내를 살해한 혐의로 재판을 받고 있습니다. 그는 자신을 변호하기 위해 고급 AI 재판관에게 90분 안에 무죄를 입증해야 합니다.
   - **개봉일**: 2026년 1월 20일
   - **평점**: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple**
   - **개요**: 과거의 사랑이 의문스럽게 죽고, 17세 딸이 복수를 위해 나서면서 벌어지는 이야기를 다룹니다.
   - **개봉일**: 2026년 1월 14일
   - **평점**: 7.2
   - ![포스터](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Les Orphelins**
   - **개요**: 고아원에서 자란 친구들이 우연으로 다시 만나고, 그들의 과거와 가족을 잃은 슬픔을 극복해 나가는 이야기입니다.
   - **개봉일**: 2025년 8월 20일
   - **평점**: 6.0
   - ![포스터](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **A Woman Scorned**
   - **개요**: 가족 휴가지에서 사건을 겪은 여성이 자매의 복수를 위해 위험한 길을 나서는 이야기입니다.
   - **개봉일**: 2025년 6월 9일
   - **평점**: 6.0
   - ![포스터](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)
